In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from collections import deque
import gymnasium as gym
import random
import sys
import os
import matplotlib.pyplot as plt
from loguru import logger

# === МОДЕЛИ ===

class Actor(nn.Module):
    def __init__(self, state_dim, action_dim=3):
        super(Actor, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(state_dim, 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.Flatten(),
        )

        self.cnn_out_size = 7 * 7 * 64

        self.fc = nn.Linear(self.cnn_out_size, 256)
        
        self.mean = nn.Linear(256, action_dim)
        # logstd - обучаемый параметр, отвечающий за "ширину" поиска
        self.logstd = nn.Parameter(torch.zeros(action_dim))

    def forward(self, state):
        # Нормализация
        x = state.float() / 255.0
        x = self.fc(self.cnn(x))
        outs = F.relu(x)
        
        means = self.mean(outs)
        # std не может быть отрицательным, поэтому берем экспоненту
        stds = self.logstd.exp()
        
        return means, stds

class Critic(nn.Module):
    def __init__(self, state_dim, action_dim=3):
        super(Critic, self).__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(state_dim, 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.Flatten(),
        )
        self.cnn_out_size = 7 * 7 * 64

        self.fc = nn.Sequential(
            nn.Linear(self.cnn_out_size + action_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
        )

    def forward(self, state, action):
        x = state.float() / 255.0
        cnn_features = self.cnn(x)
        x = torch.cat([cnn_features, action], dim=1)
        q_value = self.fc(x)
        return q_value

In [3]:
from collections import deque
import cv2
import gymnasium as gym
import random
import numpy as np
import torch


class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def add(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size, device):
        # 1. Взять случайные N элементов
        batch = random.sample(self.buffer, batch_size)

        # 2. Распаковать их в отдельные списки
        states, actions, rewards, next_states, dones = zip(*batch)

        # 3. Превратить в torch.Tensor и закинуть на device
        return (
            torch.FloatTensor(np.array(states)).to(device),
            torch.FloatTensor(np.array(actions)).to(device),
            torch.FloatTensor(np.array(rewards)).unsqueeze(1).to(device),
            torch.FloatTensor(np.array(next_states)).to(device),
            torch.FloatTensor(np.array(dones)).unsqueeze(1).to(device),
        )

    def __len__(self):
        return len(self.buffer)


class OUNoise:
    def __init__(self, action_dim, mu=0.0, theta=0.15, sigma=0.2):
        self.action_dim = action_dim
        self.mu = mu
        self.theta = theta
        self.sigma = sigma
        self.state = np.ones(self.action_dim) * self.mu
        self.reset()

    def reset(self):
        self.state = np.ones(self.action_dim) * self.mu

    def sample(self):
        x = self.state
        dx = self.theta * (self.mu - x) + self.sigma * np.random.randn(len(x))
        self.state = x + dx
        return self.state


class CarRacingWrapper(gym.Wrapper):
    def __init__(self, env, stack_frames=4):
        super().__init__(env)
        self.stack_frames = stack_frames
        self.frames = deque(maxlen=stack_frames)

        # На выходе будет массив (4, 84, 84) - 4 кадра по 84x84
        self.observation_space = gym.spaces.Box(
            low=0, high=255, shape=(stack_frames, 84, 84), dtype=np.uint8
        )

    def _process_frame(self, frame):
        frame = frame[:84, :, :]

        frame = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)

        frame = cv2.resize(frame, (84, 84))

        return frame

    def reset(self, **kwargs):
        state, info = self.env.reset(**kwargs)
        processed = self._process_frame(state)
        for _ in range(self.stack_frames):  # Заполняем буфер дублями первого кадра
            self.frames.append(processed)
        return np.stack(self.frames, axis=0), info

    def step(self, action):
        state, reward, terminated, truncated, info = self.env.step(action)
        processed = self._process_frame(state)
        self.frames.append(processed)
        return np.stack(self.frames, axis=0), reward, terminated, truncated, info


In [4]:
class DDPGAgent:
    def __init__(
        self,
        state_dim,
        action_dim,
        device,
        lr_actor=1e-4,
        lr_critic=1e-3,
        gamma=0.99,
        tau=0.005,
        memory_size=100_000,
    ):
        self.device = device
        self.action_dim = action_dim
        self.gamma = gamma
        self.tau = tau

        self.actor = Actor(state_dim, action_dim).to(device)
        self.critic = Critic(state_dim, action_dim).to(device)
        self.actor_target = Actor(state_dim, action_dim).to(device)
        self.critic_target = Critic(state_dim, action_dim).to(device)
        
        self.actor_target.load_state_dict(self.actor.state_dict())
        self.critic_target.load_state_dict(self.critic.state_dict())

        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=lr_actor)
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=lr_critic)

        self.memory = ReplayBuffer(memory_size)
        
        # OUNoise нам больше НЕ НУЖЕН, так как шум встроен в Actor (через std)
        # Но для совместимости оставим поле, чтобы не переписывать reset()
        class DummyNoise:
            def reset(self): pass
            sigma = 0 # Заглушка
        self.noise = DummyNoise()

    def act(self, state, add_noise=True):
        state_t = torch.FloatTensor(state).unsqueeze(0).to(self.device)

        self.actor.eval()
        with torch.no_grad():
            mu, std = self.actor(state_t)
            
            if add_noise:
                dist = torch.distributions.Normal(mu, std)
                raw_action = dist.sample()
            else:
                raw_action = mu

        self.actor.train()
        
        # Squash (сжимаем в диапазоны)
        final_action = self._squash_action(raw_action)
        return final_action.cpu().numpy()[0]
        
    def _squash_action(self, raw_action):
        """Вспомогательная функция для активаций"""
        final_action = torch.zeros_like(raw_action)
        final_action[..., 0] = torch.tanh(raw_action[..., 0]) # Steering
        final_action[..., 1] = torch.sigmoid(raw_action[..., 1]) # Gas
        final_action[..., 2] = torch.sigmoid(raw_action[..., 2]) # Brake
        return final_action

    def learn(self, batch_size):
        if len(self.memory) < batch_size:
            return

        states, actions, rewards, next_states, dones = self.memory.sample(
            batch_size, self.device
        )

        # --- 1. Обновление Critic ---
        with torch.no_grad():
            # Получаем распределение для следующего шага от Target сети
            next_mu, next_std = self.actor_target(next_states)
            
            # Важный момент: DDPG обычно берет mu. Но раз у нас стахастика,
            # мы можем взять сэмпл (как в SAC) или просто mu.
            # Давай брать mu для стабильности цели (как в DDPG).
            # Если брать sample, это будет ближе к SAC.
            # Попробуем вариант препода: сэмплирование.
            # Но для Target лучше стабильность -> берем next_mu.
            next_raw_action = next_mu 
            
            next_actions = self._squash_action(next_raw_action)

            target_Q_values = self.critic_target(next_states, next_actions)
            target_Q = rewards + (self.gamma * target_Q_values * (1 - dones))

        current_Q = self.critic(states, actions)
        critic_loss = F.mse_loss(current_Q, target_Q)

        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        self.critic_optimizer.step()

        # --- 2. Обновление Actor ---
        
        # Получаем "сырое" действие от Актора (с градиентами!)
        # Тут мы используем reparameterization trick, который встроен в rsample()
        # Но Normal.sample() в PyTorch не дифференцируем!
        # Нам нужно rsample() (reparameterized sample).
        
        mu, std = self.actor(states)
        dist = torch.distributions.Normal(mu, std)
        
        # ВАЖНО: используем rsample, чтобы градиент прошел сквозь сэмплирование
        raw_action_pred = dist.rsample() 
        
        actions_pred = self._squash_action(raw_action_pred)

        # Максимизируем Q
        actor_loss = -self.critic(states, actions_pred).mean()

        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        self.actor_optimizer.step()

        self.soft_update(self.actor, self.actor_target)
        self.soft_update(self.critic, self.critic_target)

    def soft_update(self, local_model, target_model):
        for local_param, target_param in zip(
            local_model.parameters(), target_model.parameters()
        ):
            target_param.data.copy_(
                self.tau * local_param.data + (1.0 - self.tau) * target_param.data
            )

In [7]:
import os
import sys
import numpy as np
import gymnasium as gym
from loguru import logger
import torch
import matplotlib.pyplot as plt
from collections import deque

# from agent import DDPGAgent
# from utils import CarRacingWrapper

# Конфиг
BATCH_SIZE = 64
LR_ACTOR = 1e-4
LR_CRITIC = 1e-3
GAMMA = 0.99
TAU = 0.005
MEMORY_SIZE = 100_000
EPISODES = 1000
MAX_STEPS = 1000
TARGET_SCORE = 500
# CHECKPOINT_DIR = os.path.dirname(os.path.abspath(__file__))
CHECKPOINT_DIR = "/Users/nabandurko/repos/otus-rl/ddpg_car_racing"
LOG_FILE = os.path.join(CHECKPOINT_DIR, "ddpg_training.log")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)


def setup_logging():
    """Логи в консоль (stderr) и в файл."""
    logger.remove()
    fmt_console = "<green>{time:HH:mm:ss}</green> | <level>{level: <8}</level> | <level>{message}</level>"
    fmt_file = "{time:YYYY-MM-DD HH:mm:ss} | {level: <8} | {message}"
    logger.add(sys.stderr, format=fmt_console, level="INFO")
    logger.add(
        LOG_FILE,
        format=fmt_file,
        level="DEBUG",
        rotation="10 MB",
        retention="3 days",
    )
    logger.info(f"Logging: console + file {LOG_FILE}")


def train():
    setup_logging()

    env = gym.make(
        "CarRacing-v3",
        continuous=True,
        render_mode="rgb_array",
        lap_complete_percent=0.95,
        domain_randomize=False,
    )
    env = CarRacingWrapper(env, stack_frames=4)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"Device: {device}")

    agent = DDPGAgent(
        state_dim=4,
        action_dim=3,
        device=device,
        lr_actor=LR_ACTOR,
        lr_critic=LR_CRITIC,
        tau=TAU,
        memory_size=MEMORY_SIZE,
    )

    best_score = -100
    scores = []
    scores_window = deque(maxlen=100)

    try:
        for i_episode in range(1, EPISODES + 1):
            state, _ = env.reset()
            score = 0
            agent.noise.reset()

            for t in range(MAX_STEPS):
                action = agent.act(state, add_noise=True)
                next_state, reward, terminated, truncated, _ = env.step(action)
                done = terminated or truncated

                agent.memory.add(state, action, reward, next_state, done)
                agent.learn(BATCH_SIZE)

                state = next_state
                score += reward

                if done:
                    break

            scores_window.append(score)
            scores.append(score)
            avg_score = np.mean(scores_window)

            print(f"\rEpisode {i_episode}\tScore: {score:.2f}\tAvg: {avg_score:.2f}", end="", flush=True)

            if i_episode % 20 == 0:
                print(f"\rEpisode {i_episode}\tScore: {score:.2f}\tAvg: {avg_score:.2f}")
                logger.info(f"Episode {i_episode}, Avg: {avg_score:.2f}")
                torch.save(
                    agent.actor.state_dict(),
                    os.path.join(CHECKPOINT_DIR, "actor_checkpoint.pth"),
                )
                torch.save(
                    agent.critic.state_dict(),
                    os.path.join(CHECKPOINT_DIR, "critic_checkpoint.pth"),
                )

            if avg_score > best_score:
                best_score = avg_score
                torch.save(
                    agent.actor.state_dict(),
                    os.path.join(CHECKPOINT_DIR, "best_actor.pth"),
                )
                torch.save(
                    agent.critic.state_dict(),
                    os.path.join(CHECKPOINT_DIR, "best_critic.pth"),
                )
                logger.info(f"New Best Score: {best_score:.2f} -> Model Saved!")

            if avg_score >= TARGET_SCORE:
                logger.success(
                    f"Solved! Average score over 100 episodes: {avg_score:.2f}"
                )
                torch.save(
                    agent.actor.state_dict(),
                    os.path.join(CHECKPOINT_DIR, "solved_actor.pth"),
                )
                break

            agent.noise.sigma = max(0.1, agent.noise.sigma * 0.999)

        if np.mean(scores_window) < TARGET_SCORE:
            print()

    except KeyboardInterrupt:
        print("\nInterrupted.")
        logger.warning("Training interrupted by user")

    plt.figure(figsize=(10, 6))
    plt.plot(scores, alpha=0.3, color="cyan")
    if len(scores) >= 100:
        avg = np.convolve(scores, np.ones(100) / 100, mode="valid")
        plt.plot(range(99, len(scores)), avg, color="blue", linewidth=2, label="MA(100)")
    plt.ylabel("Score")
    plt.xlabel("Episode")
    plt.legend()
    plt.savefig(os.path.join(CHECKPOINT_DIR, "ddpg_scores.png"))
    plt.close()
    logger.info("Plot saved to ddpg_scores.png")

    return scores

In [ ]:
train()

15:27:07 | INFO     | Logging: console + file /Users/nabandurko/repos/otus-rl/ddpg_car_racing/ddpg_training.log
objc[97026]: Class SDLApplication is implemented in both /Users/nabandurko/repos/otus-rl/.venv/lib/python3.10/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x17d7cc890) and /Users/nabandurko/repos/otus-rl/.venv/lib/python3.10/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x17fe712c8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[97026]: Class SDLAppDelegate is implemented in both /Users/nabandurko/repos/otus-rl/.venv/lib/python3.10/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x17d7cc8e0) and /Users/nabandurko/repos/otus-rl/.venv/lib/python3.10/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x17fe71318). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[97026]: Class SDLTranslatorResponder is implemented in both /Users/naba

Episode 1	Score: -78.72	Avg: -78.72

15:29:14 | INFO     | New Best Score: -78.72 -> Model Saved!


Episode 4	Score: -46.49	Avg: -71.98

15:35:47 | INFO     | New Best Score: -71.98 -> Model Saved!


...

Episode 100	Score: -93.42	Avg: -84.89
Episode 120	Score: -93.40	Avg: -89.85

19:48:03 | INFO     | Episode 120, Avg: -89.85


Episode 120	Score: -93.40	Avg: -89.85
Episode 140	Score: -93.06	Avg: -93.12

20:31:30 | INFO     | Episode 140, Avg: -93.12


Episode 140	Score: -93.06	Avg: -93.12
Episode 159	Score: -93.29	Avg: -93.14